In [1]:
import random
import torch
import pandas as pd
from convert_data_to_unimorph import get_unimorph_to_vabamorf_dict, convert_saldom_to_dict
from split_data import filter_data, get_data_matrices_lexical, get_data_matrices_structural, \
                        get_train_test_split_dbca, get_atom_compound_div, get_train_test_split_systematicity, \
                        get_train_test_split_productivity, get_random_split, prepare_substitutivity_test, \
                        get_atom_synonym_counts, get_train_test_df, get_df_test_localism, apply_exception

/home/x_katku/.conda/envs/cross_lingual_generalization_mamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
#np.random.seed(SEED)

In [3]:
df_est = pd.read_excel("Estonian_dev_processed_corpus_v2.xlsx")

In [4]:
df_est.shape

(9307, 4)

In [5]:
df_swe = pd.read_excel("Swedish_dev_processed_corpus.xlsx")

In [6]:
df_swe.shape

(9172, 4)

In [7]:
#6 different splits

# 1. DBCA split (but I think I should be able to calculate DBCA scores for most/all splits)
# 2. Systematicity (Evaluating on pairs of functions/operations that were never seen together)
# 3. Productivity (length generalization)
# 4. Substitutivity (synonyms)
# 5. Localism (generating the output sequence item by item instead of all at once)
# 6. Overgeneralisation (exceptions)

In [8]:
# Lexical
# Atoms = lemmas & morphological features
# Compounds = their combinations

In [9]:
# Structural
# Atoms = morphological features
# Compounds = their combinations

In [10]:
TEST_RATIO = 0.3
TRAIN_SET_ID = 0
TEST_SET_ID = 1
VAL_RATIO_FROM_TEST = 0.5
VAL_SET_ID = 2

<h2>Filter data & get data matrices</h2>

In [11]:
df_est_after_filter = filter_data(df_est, "Unimorph_features_lemma", "Inflected_forms", "Original_tokens", "Original_text")

Removing the following feature combinations with weight lower than 0.25: {'V;ACT;PRS;POS;IMP;3;SG|V;ACT;PRS;POS;IMP;3;PL', 'V;ACT;PST;POS;IND;2;PL', 'ADJ;SPRL;AT+ALL;PL', 'ADJ;SPRL;IN+ABL;SG', 'ADJ;POS;TERM;SG', 'ADJ;CMPR;IN+ALL;SG', 'ADJ;SPRL;IN+ESS;SG', 'ADJ;CMPR;IN+ABL;PL', 'V;ACT;PRS;POS;IND;2;PL', 'ADJ;SPRL;AT+ESS;PL', 'ADJ;CMPR;AT+ESS;PL', 'ADJ;SPRL;IN+ESS;PL', 'ADJ;POS;COM;SG'}


In [12]:
df_swe_after_filter = filter_data(df_swe, "Unimorph_features_lemma", "Inflected_forms", "Original_tokens", "Original_text")

Removing the following feature combinations with weight lower than 0.25: set()


In [13]:
atom_freq_mat_struct_est, com_freq_mat_struct_est, atom_ids_struct_est, compound_ids_struct_est = get_data_matrices_structural(df_est_after_filter, "Unimorph_features_lemma")

Number of different atoms: 36
Total number of atoms: 351868
Number of different compounds: 96
Total number of compounds: 98657


In [14]:
atom_freq_mat_struct_swe, com_freq_mat_struct_swe, atom_ids_struct_swe, compound_ids_struct_swe = get_data_matrices_structural(df_swe_after_filter, "Unimorph_features_lemma")

Number of different atoms: 23
Total number of atoms: 344304
Number of different compounds: 27
Total number of compounds: 86643


In [15]:
#Decided to not treat lemmas as atoms, they are instead treated as variables

In [47]:
#atom_freq_mat_lex_est, com_freq_mat_lex_est, atom_ids_lex_est, compound_ids_lex_est = get_data_matrices_lexical(df_est_after_filter, "Unimorph_features_lemma")

Number of different atoms: 11972
Total number of atoms: 450525
Number of different compounds: 25428
Total number of compounds: 98657


In [48]:
#atom_freq_mat_lex_swe, com_freq_mat_lex_swe, atom_ids_lex_swe, compound_ids_lex_swe = get_data_matrices_lexical(df_swe_after_filter, "Unimorph_features_lemma")

Number of different atoms: 10295
Total number of atoms: 430947
Number of different compounds: 16790
Total number of compounds: 86643


<h2>Get random splits</h2>

In [18]:
split_idx_random_est = get_random_split(df_est_after_filter.shape[0], test_ratio=TEST_RATIO, train_set_id = TRAIN_SET_ID, test_set_id=TEST_SET_ID)
split_idx_random_swe = get_random_split(df_swe_after_filter.shape[0], test_ratio=TEST_RATIO, train_set_id = TRAIN_SET_ID, test_set_id=TEST_SET_ID)

In [19]:
split_idx_random_est[split_idx_random_est == 1].shape

torch.Size([2791])

In [20]:
split_idx_random_est[split_idx_random_est == 2].shape

torch.Size([0])

In [21]:
df_train_random_est, df_test_random_est = get_train_test_df(df_est_after_filter, split_idx_random_est, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)
split_idx_test_random_est = get_random_split(df_test_random_est.shape[0], test_ratio=VAL_RATIO_FROM_TEST, train_set_id = VAL_SET_ID, test_set_id=TEST_SET_ID)
df_test1_random_est, df_test2_random_est = get_train_test_df(df_test_random_est, split_idx_test_random_est, train_set_id = VAL_SET_ID, test_set_id = TEST_SET_ID)

In [22]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_random_est)

(8.124113082885742e-05, 0.0003167390823364258)

In [23]:
split_idx_random_est[(split_idx_random_est == TEST_SET_ID).nonzero().squeeze()] = split_idx_test_random_est

In [24]:
split_idx_random_est[split_idx_random_est == 1].shape

torch.Size([1395])

In [25]:
split_idx_random_est[split_idx_random_est == 2].shape

torch.Size([1396])

In [26]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_random_est, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)

(0.00015991926193237305, 0.00046128034591674805)

In [27]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_random_est, train_set_id = TRAIN_SET_ID, test_set_id = VAL_SET_ID)

(0.00010275840759277344, 0.0005384683609008789)

In [28]:
df_train_random_swe, df_test_random_swe = get_train_test_df(df_swe_after_filter, split_idx_random_swe, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)
split_idx_test_random_swe = get_random_split(df_test_random_swe.shape[0], test_ratio=VAL_RATIO_FROM_TEST, train_set_id = VAL_SET_ID, test_set_id=TEST_SET_ID)
df_test1_random_swe, df_test2_random_swe = get_train_test_df(df_test_random_swe, split_idx_test_random_swe, train_set_id = VAL_SET_ID, test_set_id = TEST_SET_ID)

In [29]:
get_atom_compound_div(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, split_idx_random_swe)

(6.16312026977539e-05, 8.797645568847656e-05)

In [30]:
split_idx_random_swe[(split_idx_random_swe == TEST_SET_ID).nonzero().squeeze()] = split_idx_test_random_swe

In [31]:
get_atom_compound_div(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, split_idx_random_swe, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)

(9.810924530029297e-05, 0.0001372694969177246)

In [32]:
get_atom_compound_div(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, split_idx_random_swe, train_set_id = TRAIN_SET_ID, test_set_id = VAL_SET_ID)

(9.304285049438477e-05, 0.0001628398895263672)

In [33]:
df_train_random_est.shape, df_test1_random_est.shape, df_test2_random_est.shape

((6515, 4), (1396, 4), (1395, 4))

In [34]:
df_train_random_swe.shape, df_test1_random_swe.shape, df_test2_random_swe.shape

((6421, 4), (1376, 4), (1375, 4))

<h2>1. DBCA</h2>

In [53]:
split_idx_struct_est = get_train_test_split_dbca(atom_freq_mat_struct_est, com_freq_mat_struct_est,
                                      target_com_div=1.0, min_test_percent=0.25,
                                      max_test_percent=0.35, subsample_size = 10000, device="cuda")
split_idx_struct_est = split_idx_struct_est.to("cpu")
torch.cuda.empty_cache()

Initialised the train set with one random sentence.
Starting division. Train set size: 1, Test set size: 0.
Dividing 9306 sentences...


100%|█████████████████████████████████████████████████████████████████████████| 9305/9305 [00:24<00:00, 382.73it/s]


In [54]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_struct_est)

(0.012868344783782959, 0.039031267166137695)

In [55]:
df_dbca_struct_train_est, df_dbca_struct_test_est = get_train_test_df(df_est_after_filter, split_idx_struct_est, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)
split_idx_test_struct_est = get_random_split(df_dbca_struct_test_est.shape[0], test_ratio=VAL_RATIO_FROM_TEST, train_set_id = VAL_SET_ID, test_set_id=TEST_SET_ID)
df_dbca_test1_struct_est, df_dbca_test2_struct_est = get_train_test_df(df_dbca_struct_test_est, split_idx_test_struct_est, train_set_id = VAL_SET_ID, test_set_id = TEST_SET_ID)

In [56]:
split_idx_struct_est[(split_idx_struct_est == TEST_SET_ID).nonzero().squeeze()] = split_idx_test_struct_est

In [57]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_struct_est, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)

(0.01289135217666626, 0.039545655250549316)

In [58]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_struct_est, train_set_id = TRAIN_SET_ID, test_set_id = VAL_SET_ID)

(0.01311182975769043, 0.039435744285583496)

In [59]:
split_idx_struct_swe = get_train_test_split_dbca(atom_freq_mat_struct_swe, com_freq_mat_struct_swe,
                                      target_com_div=1.0, min_test_percent=0.25,
                                      max_test_percent=0.35, subsample_size = 10000, device="cuda")
split_idx_struct_swe = split_idx_struct_swe.to("cpu")
torch.cuda.empty_cache()

Initialised the train set with one random sentence.
Starting division. Train set size: 1, Test set size: 0.
Dividing 9172 sentences...


100%|█████████████████████████████████████████████████████████████████████████| 9171/9171 [00:23<00:00, 394.95it/s]


In [60]:
get_atom_compound_div(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, split_idx_struct_swe)

(0.008712410926818848, 0.020855963230133057)

In [61]:
df_dbca_struct_train_swe, df_dbca_struct_test_swe = get_train_test_df(df_swe_after_filter, split_idx_struct_swe, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)
split_idx_test_struct_swe = get_random_split(df_dbca_struct_test_swe.shape[0], test_ratio=VAL_RATIO_FROM_TEST, train_set_id = VAL_SET_ID, test_set_id=TEST_SET_ID)
df_dbca_test1_struct_swe, df_dbca_test2_struct_swe = get_train_test_df(df_dbca_struct_test_swe, split_idx_test_struct_swe, train_set_id = VAL_SET_ID, test_set_id = TEST_SET_ID)

In [62]:
split_idx_struct_swe[(split_idx_struct_swe == TEST_SET_ID).nonzero().squeeze()] = split_idx_test_struct_swe

In [63]:
get_atom_compound_div(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, split_idx_struct_swe, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)

(0.00864642858505249, 0.020667731761932373)

In [64]:
get_atom_compound_div(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, split_idx_struct_swe, train_set_id = TRAIN_SET_ID, test_set_id = VAL_SET_ID)

(0.008789777755737305, 0.02109825611114502)

In [65]:
df_dbca_struct_train_est.shape, df_dbca_test1_struct_est.shape, df_dbca_test2_struct_est.shape

((6844, 4), (1231, 4), (1231, 4))

In [66]:
df_dbca_struct_train_swe.shape, df_dbca_test1_struct_swe.shape, df_dbca_test2_struct_swe.shape

((5964, 4), (1604, 4), (1604, 4))

<h2>2. Systematicity </h2>

In [76]:
split_idx_syst_est = get_train_test_split_systematicity(atom_freq_mat_struct_est, com_freq_mat_struct_est, [["V", "PASS", "PRS", "POS", "IND"], ["N", "IN+ABL", "PL"], ["ADJ", "SPRL", "PRT", "SG"]], atom_ids_struct_est, compound_ids_struct_est)

All atoms in ['V', 'PASS', 'PRS', 'POS', 'IND'] present in the train set.
All atoms in ['N', 'IN+ABL', 'PL'] present in the train set.
All atoms in ['ADJ', 'SPRL', 'PRT', 'SG'] present in the train set.
Train size: 7303
Test size: 2003


In [77]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_syst_est)

(0.005949795246124268, 0.0795363187789917)

In [78]:
df_syst_train_est, df_syst_test_est = get_train_test_df(df_est_after_filter, split_idx_syst_est, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)
split_idx_test_syst_est = get_random_split(df_syst_test_est.shape[0], test_ratio=VAL_RATIO_FROM_TEST, train_set_id = VAL_SET_ID, test_set_id=TEST_SET_ID)
df_test1_syst_est, df_test2_syst_est = get_train_test_df(df_syst_test_est, split_idx_test_syst_est, train_set_id = VAL_SET_ID, test_set_id = TEST_SET_ID)

In [79]:
split_idx_syst_est[(split_idx_syst_est == TEST_SET_ID).nonzero().squeeze()] = split_idx_test_syst_est

In [80]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_syst_est, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)

(0.0060561299324035645, 0.08031868934631348)

In [81]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_syst_est, train_set_id = TRAIN_SET_ID, test_set_id = VAL_SET_ID)

(0.00592881441116333, 0.0790867805480957)

In [82]:
split_idx_syst_swe = get_train_test_split_systematicity(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, [["V", "IND", "PASS", "PST"], ["N", "GEN", "PL", "DEF"], ["ADJ", "POS", "MASC+FEM", "SG", "INDF"]], atom_ids_struct_swe, compound_ids_struct_swe)

All atoms in ['V', 'IND', 'PASS', 'PST'] present in the train set.
All atoms in ['N', 'GEN', 'PL', 'DEF'] present in the train set.
All atoms in ['ADJ', 'POS', 'MASC+FEM', 'SG', 'INDF'] present in the train set.
Train size: 6993
Test size: 2179


In [83]:
get_atom_compound_div(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, split_idx_syst_swe)

(0.009079217910766602, 0.08659762144088745)

In [84]:
df_syst_train_swe, df_syst_test_swe = get_train_test_df(df_swe_after_filter, split_idx_syst_swe, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)
split_idx_test_syst_swe = get_random_split(df_syst_test_swe.shape[0], test_ratio=VAL_RATIO_FROM_TEST, train_set_id = VAL_SET_ID, test_set_id=TEST_SET_ID)
df_test1_syst_swe, df_test2_syst_swe = get_train_test_df(df_syst_test_swe, split_idx_test_syst_swe, train_set_id = VAL_SET_ID, test_set_id = TEST_SET_ID)

In [85]:
df_syst_train_est.shape, df_test1_syst_est.shape, df_test2_syst_est.shape

((7303, 4), (1002, 4), (1001, 4))

In [86]:
df_syst_train_swe.shape, df_test1_syst_swe.shape, df_test2_syst_swe.shape

((6993, 4), (1090, 4), (1089, 4))

<h2>3. Productivity</h2>

In [90]:
split_idx_prod_est = get_train_test_split_productivity(atom_freq_mat_struct_est, com_freq_mat_struct_est, atom_ids_struct_est, compound_ids_struct_est, sentence_level=True, max_train_len=12)

Train size: 6215
Test size: 3091


In [91]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_prod_est)

(0.003379344940185547, 0.0049468278884887695)

In [92]:
df_prod_train_est, df_prod_test_est = get_train_test_df(df_est_after_filter, split_idx_prod_est, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)
split_idx_test_prod_est = get_random_split(df_prod_test_est.shape[0], test_ratio=VAL_RATIO_FROM_TEST, train_set_id = VAL_SET_ID, test_set_id=TEST_SET_ID)
df_test1_prod_est, df_test2_prod_est = get_train_test_df(df_prod_test_est, split_idx_test_prod_est, train_set_id = VAL_SET_ID, test_set_id = TEST_SET_ID)

In [93]:
split_idx_prod_est[(split_idx_prod_est == TEST_SET_ID).nonzero().squeeze()] = split_idx_test_prod_est

In [94]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_prod_est, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)

(0.0032888054847717285, 0.0049266815185546875)

In [95]:
get_atom_compound_div(atom_freq_mat_struct_est, com_freq_mat_struct_est, split_idx_prod_est, train_set_id = TRAIN_SET_ID, test_set_id = VAL_SET_ID)

(0.003512084484100342, 0.00511932373046875)

In [96]:
split_idx_prod_swe = get_train_test_split_productivity(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, atom_ids_struct_swe, compound_ids_struct_swe, sentence_level=True, max_train_len=12)

Train size: 6647
Test size: 2525


In [97]:
get_atom_compound_div(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, split_idx_prod_swe)

(0.0013953447341918945, 0.0015984773635864258)

In [98]:
df_prod_train_swe, df_prod_test_swe = get_train_test_df(df_swe_after_filter, split_idx_prod_swe, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)
split_idx_test_prod_swe = get_random_split(df_prod_test_swe.shape[0], test_ratio=VAL_RATIO_FROM_TEST, train_set_id = VAL_SET_ID, test_set_id=TEST_SET_ID)
df_test1_prod_swe, df_test2_prod_swe = get_train_test_df(df_prod_test_swe, split_idx_test_prod_swe, train_set_id = VAL_SET_ID, test_set_id = TEST_SET_ID)

In [99]:
split_idx_prod_swe[(split_idx_prod_swe == TEST_SET_ID).nonzero().squeeze()] = split_idx_test_prod_swe

In [100]:
get_atom_compound_div(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, split_idx_prod_swe, train_set_id = TRAIN_SET_ID, test_set_id = TEST_SET_ID)

(0.0016194581985473633, 0.0016939640045166016)

In [101]:
get_atom_compound_div(atom_freq_mat_struct_swe, com_freq_mat_struct_swe, split_idx_prod_swe, train_set_id = TRAIN_SET_ID, test_set_id = VAL_SET_ID)

(0.0012183785438537598, 0.0015978217124938965)

In [102]:
df_prod_train_est.shape, df_test1_prod_est.shape, df_test2_prod_est.shape

((6215, 4), (1546, 4), (1545, 4))

In [103]:
df_prod_train_swe.shape, df_test1_prod_swe.shape, df_test2_prod_swe.shape

((6647, 4), (1263, 4), (1262, 4))

<h2>4. Substitutivity (synonyms) </h2>

In [107]:
df_train_sub_est, df_test_orig_sub_est1, df_test_syn_sub_est1 = prepare_substitutivity_test(df_est_after_filter, "Unimorph_features_lemma", split_idx_random_est, {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})
_, df_test_orig_sub_est2, df_test_syn_sub_est2 = prepare_substitutivity_test(df_est_after_filter, "Unimorph_features_lemma", split_idx_random_est, {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"}, test_set_id = 2)

/proj/dl4nlp/users/x_katku/projects/cross_lingual_compositional_generalization/data/split_data.py:760: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train[input_column_name] = df_train.apply(lambda x: replace_atoms_with_synonyms(x[input_column_name],


In [108]:
get_atom_synonym_counts(df_train_sub_est, "Unimorph_features_lemma", {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})

{('GEN', 10676): ('ENAF', 10540),
 ('CMPR', 98): ('PAH', 141),
 ('PASS', 2563): ('EDLG', 2506),
 ('SG', 23145): ('TEK', 23379)}

In [109]:
get_atom_synonym_counts(df_test_orig_sub_est1, "Unimorph_features_lemma", {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})

{('GEN', 4549): ('ENAF', 0),
 ('CMPR', 41): ('PAH', 0),
 ('PASS', 1023): ('EDLG', 0),
 ('SG', 9755): ('TEK', 0)}

In [110]:
get_atom_synonym_counts(df_test_syn_sub_est1, "Unimorph_features_lemma", {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})

{('GEN', 0): ('ENAF', 4549),
 ('CMPR', 0): ('PAH', 41),
 ('PASS', 0): ('EDLG', 1023),
 ('SG', 0): ('TEK', 9755)}

In [111]:
get_atom_synonym_counts(df_test_orig_sub_est2, "Unimorph_features_lemma", {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})

{('GEN', 4441): ('ENAF', 0),
 ('CMPR', 51): ('PAH', 0),
 ('PASS', 1018): ('EDLG', 0),
 ('SG', 9784): ('TEK', 0)}

In [112]:
get_atom_synonym_counts(df_test_syn_sub_est2, "Unimorph_features_lemma", {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})

{('GEN', 0): ('ENAF', 4441),
 ('CMPR', 0): ('PAH', 51),
 ('PASS', 0): ('EDLG', 1018),
 ('SG', 0): ('TEK', 9784)}

In [113]:
df_train_sub_est.shape, df_test_orig_sub_est1.shape, df_test_syn_sub_est1.shape, df_test_orig_sub_est2.shape, df_test_syn_sub_est2.shape

((6515, 4), (1395, 4), (1395, 4), (1396, 4), (1396, 4))

In [114]:
df_train_sub_swe, df_test_orig_sub_swe1, df_test_syn_sub_swe1 = prepare_substitutivity_test(df_swe_after_filter, "Unimorph_features_lemma", split_idx_random_swe, {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})
_, df_test_orig_sub_swe2, df_test_syn_sub_swe2 = prepare_substitutivity_test(df_swe_after_filter, "Unimorph_features_lemma", split_idx_random_swe, {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"}, test_set_id=2)

/proj/dl4nlp/users/x_katku/projects/cross_lingual_compositional_generalization/data/split_data.py:760: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train[input_column_name] = df_train.apply(lambda x: replace_atoms_with_synonyms(x[input_column_name],


In [115]:
get_atom_synonym_counts(df_train_sub_swe, "Unimorph_features_lemma", {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})

{('GEN', 932): ('ENAF', 928),
 ('CMPR', 183): ('PAH', 143),
 ('PASS', 1906): ('EDLG', 1853),
 ('SG', 15016): ('TEK', 14825)}

In [116]:
get_atom_synonym_counts(df_test_orig_sub_swe1, "Unimorph_features_lemma", {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})

{('GEN', 401): ('ENAF', 0),
 ('CMPR', 75): ('PAH', 0),
 ('PASS', 836): ('EDLG', 0),
 ('SG', 6689): ('TEK', 0)}

In [117]:
get_atom_synonym_counts(df_test_syn_sub_swe1, "Unimorph_features_lemma", {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})

{('GEN', 0): ('ENAF', 401),
 ('CMPR', 0): ('PAH', 75),
 ('PASS', 0): ('EDLG', 836),
 ('SG', 0): ('TEK', 6689)}

In [118]:
get_atom_synonym_counts(df_test_orig_sub_swe2, "Unimorph_features_lemma", {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})

{('GEN', 416): ('ENAF', 0),
 ('CMPR', 55): ('PAH', 0),
 ('PASS', 836): ('EDLG', 0),
 ('SG', 6453): ('TEK', 0)}

In [119]:
get_atom_synonym_counts(df_test_syn_sub_swe2, "Unimorph_features_lemma", {"GEN" : "ENAF", "CMPR" : "PAH", "PASS" : "EDLG", "SG" : "TEK"})

{('GEN', 0): ('ENAF', 416),
 ('CMPR', 0): ('PAH', 55),
 ('PASS', 0): ('EDLG', 836),
 ('SG', 0): ('TEK', 6453)}

In [120]:
df_train_sub_swe.shape, df_test_orig_sub_swe1.shape, df_test_syn_sub_swe1.shape, df_test_orig_sub_swe2.shape, df_test_syn_sub_swe2.shape

((6421, 4), (1375, 4), (1375, 4), (1376, 4), (1376, 4))

In [121]:
df_train_sub_est.shape, df_test_orig_sub_est1.shape, df_test_syn_sub_est1.shape, df_test_orig_sub_est2.shape, df_test_syn_sub_est2.shape

((6515, 4), (1395, 4), (1395, 4), (1396, 4), (1396, 4))

<h2>5. Localism</h2>

In [125]:
df_test1_loc_est = get_df_test_localism(df_test1_random_est)
df_test2_loc_est = get_df_test_localism(df_test2_random_est)

In [126]:
df_test1_loc_swe = get_df_test_localism(df_test1_random_swe)
df_test2_loc_swe = get_df_test_localism(df_test2_random_swe)

In [127]:
df_test1_loc_est.shape, df_test2_loc_est.shape

((1396, 4), (1395, 4))

In [128]:
df_test1_loc_swe.shape, df_test2_loc_swe.shape

((1376, 4), (1375, 4))

<h2>6. Overgeneralisation</h2>

In [ ]:
# Example: Map SG to PL if SG appears together with PRS+POS
# exceptions_dict = {"SG" : {"value" : "PL", "condition" : ["V", ""PRS", "POS"]}}

In [132]:
exceptions_dict_est = {"SG" : {"value" : "PL", "condition" : ["V", "IND", "PRS"]}, "FRML" : {"value" : "TRANS", "condition" : ["N", "SG"]}}

In [133]:
exceptions_dict_swe = {"SG" : {"value" : "PL", "condition" : ["N", "GEN", "INDF"]}, "ACT" : {"value" : "PASS", "condition" : ["V", "NFIN"]}}

In [135]:
conversion_dict_est = get_unimorph_to_vabamorf_dict("./from_eesthetic/estonian_cells.csv")

In [136]:
conversion_dict_swe = convert_saldom_to_dict("./SALDO/saldom.xml")

In [137]:
df_train_except_est = apply_exception(df_train_random_est, "Unimorph_features_lemma", "Inflected_forms", exceptions_dict_est, "Estonian", conversion_dict_est)
df_test1_except_est = apply_exception(df_test1_random_est, "Unimorph_features_lemma", "Inflected_forms", exceptions_dict_est, "Estonian", conversion_dict_est)
df_test2_except_est = apply_exception(df_test2_random_est, "Unimorph_features_lemma", "Inflected_forms", exceptions_dict_est, "Estonian", conversion_dict_est)

SG&V&IND&PRS appears in 3811 items.
FRML&N&SG appears in 233 items.
SG&V&IND&PRS appears in 756 items.
FRML&N&SG appears in 35 items.
SG&V&IND&PRS appears in 759 items.
FRML&N&SG appears in 43 items.


In [138]:
df_train_except_swe = apply_exception(df_train_random_swe, "Unimorph_features_lemma", "Inflected_forms", exceptions_dict_swe, "Swedish", conversion_dict_swe)
df_test1_except_swe = apply_exception(df_test1_random_swe, "Unimorph_features_lemma", "Inflected_forms", exceptions_dict_swe, "Swedish", conversion_dict_swe)
df_test2_except_swe = apply_exception(df_test2_random_swe, "Unimorph_features_lemma", "Inflected_forms", exceptions_dict_swe, "Swedish", conversion_dict_swe)

SG&N&GEN&INDF appears in 70 items.
ACT&V&NFIN appears in 3240 items.
SG&N&GEN&INDF appears in 15 items.
ACT&V&NFIN appears in 663 items.
SG&N&GEN&INDF appears in 14 items.
ACT&V&NFIN appears in 773 items.


In [139]:
df_train_except_est.shape

(6515, 5)

In [140]:
df_train_except_swe.shape

(6421, 5)

<h2>Create duplicates of all files with codes for each morphological feature</h2>

In [143]:
all_feats = set(list(atom_ids_struct_est.keys()) + list(atom_ids_struct_swe.keys()))
unique_feats = set()
for feat in all_feats:
    if "+" in feat:
        items = feat.split("+")
        for item in items:
            unique_feats.add(item)
    else:
        unique_feats.add(feat)
        
unique_feats = sorted(list(unique_feats))
print(unique_feats)
num_codes = len(unique_feats)
print(num_codes)

['1', '2', '3', 'ABL', 'ACT', 'ADJ', 'ALL', 'AT', 'CMPR', 'COM', 'COND', 'DEF', 'ESS', 'FEM', 'FRML', 'GEN', 'IMP', 'IN', 'IND', 'INDF', 'MASC', 'N', 'NEG', 'NEUT', 'NFIN', 'NOM', 'PASS', 'PL', 'POS', 'PRIV', 'PRS', 'PRT', 'PST', 'QUOT', 'SG', 'SPRL', 'TERM', 'TRANS', 'V', 'V.CVB', 'V.PTCP']
41


In [167]:
import string

def generate_codename():
    # Pick a random length between 2 and 4
    length = random.randint(2, 4)
    # Generate the random string
    return ''.join(random.choices(string.ascii_uppercase, k=length))

#codenames = [generate_codename() for _ in range(num_codes)]
#print(codenames)

['NTM', 'QF', 'YJMK', 'ETW', 'GORZ', 'SYQB', 'DHS', 'GP', 'ZJMZ', 'DX', 'BEVL', 'JHW', 'BVQ', 'ERML', 'WJXR', 'OXPX', 'ILI', 'QE', 'IIR', 'MY', 'DV', 'IA', 'ZXUK', 'TVW', 'QW', 'RH', 'NY', 'BTD', 'DHP', 'HJ', 'YEX', 'OPE', 'BHM', 'GCYS', 'FGC', 'KNH', 'LLCJ', 'UL', 'GMR', 'NE', 'GU']


In [144]:
#codenames = ['IF', 'RPOL', 'SW', 'AHB', 'KQF', 'CNQ', 'IO', 'MSB', 'OF', 'RPZP', 'AIHD', 'OVK', 'MREH', 'BLYW', 'YIA', 'FSSX', 'ZCOK', 'EN', 'SN', 'VX', 'WXK', 'MT', 'HVJJ', 'DM', 'FYLT', 'WTA', 'YM', 'YWV', 'HYE', 'IL', 'CGJ', 'ORO', 'YJCN', 'UGJG', 'LNK', 'LUZU', 'VKHZ', 'QWIA', 'QDO', 'TJ', 'ZWHV']
codenames = ['NTM', 'QF', 'YJMK', 'ETW', 'GORZ', 'SYQB', 'DHS', 'GP', 'ZJMZ', 'DX', 'BEVL', 'JHW', 'BVQ', 'ERML', 'WJXR', 'OXPX', 'ILI', 'QE', 'IIR', 'MY', 'DV', 'IA', 'ZXUK', 'TVW', 'QW', 'RH', 'NY', 'BTD', 'DHP', 'HJ', 'YEX', 'OPE', 'BHM', 'GCYS', 'FGC', 'KNH', 'LLCJ', 'UL', 'GMR', 'NE', 'GU']

In [145]:
len(set(codenames)) == num_codes

True

In [146]:
for item in ["ENAF", "PAH", "EDLG", "TEK"]:
    print(f"{item} in codenames:", item in codenames)

ENAF in codenames: False
PAH in codenames: False
EDLG in codenames: False
TEK in codenames: False


In [147]:
feat_to_code_dict = dict()
for feat, code in zip(unique_feats, codenames):
    feat_to_code_dict[feat] = code

In [148]:
feat_to_code_dict

{'1': 'NTM',
 '2': 'QF',
 '3': 'YJMK',
 'ABL': 'ETW',
 'ACT': 'GORZ',
 'ADJ': 'SYQB',
 'ALL': 'DHS',
 'AT': 'GP',
 'CMPR': 'ZJMZ',
 'COM': 'DX',
 'COND': 'BEVL',
 'DEF': 'JHW',
 'ESS': 'BVQ',
 'FEM': 'ERML',
 'FRML': 'WJXR',
 'GEN': 'OXPX',
 'IMP': 'ILI',
 'IN': 'QE',
 'IND': 'IIR',
 'INDF': 'MY',
 'MASC': 'DV',
 'N': 'IA',
 'NEG': 'ZXUK',
 'NEUT': 'TVW',
 'NFIN': 'QW',
 'NOM': 'RH',
 'PASS': 'NY',
 'PL': 'BTD',
 'POS': 'DHP',
 'PRIV': 'HJ',
 'PRS': 'YEX',
 'PRT': 'OPE',
 'PST': 'BHM',
 'QUOT': 'GCYS',
 'SG': 'FGC',
 'SPRL': 'KNH',
 'TERM': 'LLCJ',
 'TRANS': 'UL',
 'V': 'GMR',
 'V.CVB': 'NE',
 'V.PTCP': 'GU'}

In [149]:
all_feats_to_code_dict = dict()
for feat in all_feats:
    if "+" not in feat:
        all_feats_to_code_dict[feat] = feat_to_code_dict[feat]
    else:
        parts = feat.split("+")
        all_feats_to_code_dict[feat] = "+".join([feat_to_code_dict[p] for p in parts])

In [150]:
all_feats_to_code_dict

{'SG': 'FGC',
 'MASC+FEM': 'DV+ERML',
 'TRANS': 'UL',
 'MASC': 'DV',
 'ADJ': 'SYQB',
 'V.CVB': 'NE',
 'IMP': 'ILI',
 'V': 'GMR',
 'AT+ESS': 'GP+BVQ',
 'NEG': 'ZXUK',
 'PST': 'BHM',
 'COND': 'BEVL',
 'NFIN': 'QW',
 '3': 'YJMK',
 'TERM': 'LLCJ',
 'GEN': 'OXPX',
 'AT+ABL': 'GP+ETW',
 '2': 'QF',
 'FRML': 'WJXR',
 'QUOT': 'GCYS',
 'DEF': 'JHW',
 'CMPR': 'ZJMZ',
 '1': 'NTM',
 'PL': 'BTD',
 'N': 'IA',
 'IN+ALL': 'QE+DHS',
 'POS': 'DHP',
 'AT+ALL': 'GP+DHS',
 'IN+ESS': 'QE+BVQ',
 'INDF': 'MY',
 'NOM': 'RH',
 'NEUT': 'TVW',
 'PASS': 'NY',
 'V.PTCP': 'GU',
 'ACT': 'GORZ',
 'PRS': 'YEX',
 'COM': 'DX',
 'PRIV': 'HJ',
 'SPRL': 'KNH',
 'IN+ABL': 'QE+ETW',
 'PRT': 'OPE',
 'IND': 'IIR'}

In [151]:
from pathlib import Path
from split_data import replace_atoms_with_synonyms

In [ ]:
folder = [DATA_PATH]
data_split_files = [str(path) for path in Path(folder).rglob('*') if path.is_file()]

In [153]:
input_column = "Unimorph_features_lemma"
for filename in data_split_files:
    if "original_features" in filename and "localism_splits" not in filename and filename.endswith(".xlsx"):
        df_temp = pd.read_excel(filename).copy(deep=True)
        df_temp[input_column] = df_temp.apply(lambda x: replace_atoms_with_synonyms(x[input_column], all_feats_to_code_dict), axis=1)
        save_path = filename.replace("original_features", "code_features")
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        df_temp.to_excel(save_path, index=False)